# Mar 13/10 · 11:30 — TP v0.5 : logging et codes de sortie

Mar 13/10, TP de 12:00 à 12:25. Correction à 12:25, TP fini ou non. À 12:30, écrans fermés : QCM papier de révision. La référence v0.5 est publiée à 13:00.

## Objectif

`fleetcheck` devient utilisable en CI :

- le journal (lignes ignorées, erreurs) passe par `logging` sur stderr, et stdout ne contient plus que le résultat ;
- `-v` et `-vv` règlent la quantité de journal ;
- le code de sortie dit si le contrôle est fiable : `0` succès, `1` panne détectée, `2` erreur fatale ou ligne d'inventaire invalide.

## Mode d'emploi du notebook

- Copiez ce notebook à la racine de votre dépôt, à côté de `fleetcheck.py`. Le code s'écrit dans `fleetcheck.py`, ouvert dans un onglet de JupyterLab ; le notebook sert à essayer et à vérifier chaque étape.
- Exécutez d'abord la cellule d'outils ci-dessous. Chaque vérification affiche `OK`, ou `KO` avec ce qui était attendu.
- `%run -n fleetcheck.py` charge les fonctions du fichier enregistré sans exécuter le bloc `if __name__ == "__main__":`. N'appelez pas `main()` dans le notebook : argparse y lirait les arguments du noyau Jupyter (`-f …kernel.json`) et s'arrêterait.
- `run(...)` lance le script comme dans un terminal, avec ses arguments. Il affiche la sortie standard, la sortie d'erreur (en rouge) et le code de sortie, et les renvoie : `r.stdout`, `r.stderr`, `r.returncode`. Tout ce qui touche à la ligne de commande se vérifie avec `run(...)` : options, journal, codes de sortie. `logging.basicConfig` ne s'applique qu'une fois par noyau ; chaque `run(...)` repart d'un processus neuf.
- Les vérifications qui retirent un fichier ou en créent un remettent tout en place : ne les modifiez pas.
- **Prédire** et **Justifier** marquent les moments clés du TP. Répondez en une ou deux lignes dans la cellule `# Ma réponse :` qui suit, **avant** de lire la suite ou d'exécuter quoi que ce soit. Une prédiction fausse, notée puis comprise, vaut mieux qu'une bonne réponse recopiée : c'est le geste demandé aux QCM des CC et à la soutenance. Les réponses sont reprises à la correction.
- Rendu en fin de TP : menu *Kernel > Restart Kernel and Run All Cells*, puis commit et push de `fleetcheck.py`, `config.json`, `servers.csv`, `servers-broken.csv` et de ce notebook.

In [ ]:
# Outils du TP : exécuter cette cellule en premier, sans la modifier.
import os
import subprocess
import sys


def check(obtenu, attendu):
    if obtenu == attendu:
        print("OK")
    elif isinstance(attendu, str) and "\n" in attendu:
        print(f"KO, obtenu :\n{obtenu}\nattendu :\n{attendu}")
    else:
        print(f"KO\n  obtenu  : {obtenu!r}\n  attendu : {attendu!r}")


def check_error(fonction, *args, attendu):
    try:
        fonction(*args)
    except Exception as e:
        check(f"{type(e).__name__}: {e}", attendu)
    else:
        print(f"KO, aucune exception levée. Attendu : {attendu}")


def run(*args):
    """Lance fleetcheck.py comme dans un terminal (vu en v0.8 : subprocess)."""
    r = subprocess.run([sys.executable, "fleetcheck.py", *args], capture_output=True,
                       encoding="utf-8", env={**os.environ, "PYTHONUTF8": "1", "NO_COLOR": "1"})
    print(r.stdout, end="")
    print(r.stderr, end="", file=sys.stderr)
    print(f"[code de sortie : {r.returncode}]")
    return r

## Point de départ

Votre `fleetcheck.py` de v0.4, avec `config.json`, `servers.csv` et `servers-broken.csv` dans le même dossier.

En retard ou en panne : copiez dans votre dépôt les fichiers de `reference/` au tag `v0.4` du dépôt public (`git switch --detach v0.4` dans votre clone de consultation, voir le `README.md` à la racine).

Vérifiez avant de commencer :

In [ ]:
r = run("--env", "staging")
check(r.stdout, """2 serveurs valides, 0 invalides
par env : staging 2
par rôle : web 1, api 1
prod/web : 
""")

## Étapes

| Fin | Étape |
|---|---|
| 12:08 | 1. Remplacer les `print` d'erreur par `logging` sur stderr : `WARNING` pour une ligne invalide, `ERROR` pour une erreur fatale. `-v` passe en `INFO`, `-vv` en `DEBUG` |
| 12:15 | 2. Fonction `exit_code(statuses, invalid_count)` |
| 12:25 | 3. Erreurs fatales en `ERROR` et code `2` ; code de sortie final calculé par `exit_code` |

À 12:15, si plus d'un tiers de la salle n'a pas `exit_code`, l'enseignant l'écrit au tableau et `-vv` passe en « pour aller plus loin ».

### 1. Logging sur stderr — fin 12:08

- Un logger nommé `fleetcheck`, configuré une seule fois au début de `main()` : format `%(levelname)s %(message)s`, sortie stderr.
- Une ligne invalide donne un `WARNING` de la forme `ligne N ignorée : <erreur>`. Les lignes `invalide ligne N : ...` du résumé disparaissent : le résumé ne garde que ses 4 lignes.
- Les messages d'erreur fatale de v0.4 (`print` suivi de `sys.exit`) deviennent des `ERROR`. Le code de sortie change à l'étape 3.
- Option `-v` / `--verbose`, répétable : niveau `WARNING` sans option, `INFO` avec `-v`, `DEBUG` avec `-vv`. Écrivez au moins un message `INFO` par exécution (par exemple le nombre de lignes lues) et un message `DEBUG` par ligne valide.
- Les arguments du message sont passés séparément (`log.warning("ligne %d ignorée : %s", line, e)`), pas en f-string.

Ce qui est vérifié : le texte exact des `WARNING` de lignes invalides et des `ERROR` fatales. Pour `INFO` et `DEBUG`, le texte est libre ; seuls le niveau et le flux comptent (stderr, ligne qui commence par `INFO ` ou `DEBUG `).

**Justifier** — Si les `WARNING` restaient sur stdout, que recevrait `jq` dans `python3 fleetcheck.py --inventory servers-broken.csv --format json | jq .` ?

In [ ]:
# Ma réponse :

Vérification avec `servers-broken.csv` : stdout ne garde que le résumé, stderr porte les 3 `WARNING`.

```text
WARNING ligne 4 ignorée : port non entier : 'abc'
WARNING ligne 8 ignorée : IP invalide : '300.0.0.1'
WARNING ligne 12 ignorée : 3 champs au lieu de 5
```

In [ ]:
r = run("--inventory", "servers-broken.csv")
check(r.stdout, """9 serveurs valides, 3 invalides
par env : prod 7, staging 2
par rôle : web 3, api 3, worker 1, cache 1, db 1
prod/web : web-1, web-2
""")
check(r.stderr, """WARNING ligne 4 ignorée : port non entier : 'abc'
WARNING ligne 8 ignorée : IP invalide : '300.0.0.1'
WARNING ligne 12 ignorée : 3 champs au lieu de 5
""")

Sans option, un run sur `servers.csv` n'écrit rien sur stderr :

In [ ]:
r = run()
check(r.stderr, "")

Avec `-v`, la référence écrit sur stderr :

```text
INFO 9 serveur(s) valide(s), 0 ligne(s) invalide(s) dans servers.csv
INFO rapport écrit : report.json
```

Vérification : chaque ligne de stderr commence par un niveau, au moins une est en `INFO`, aucune en `DEBUG`, et stdout ne contient que le résumé.

In [ ]:
NIVEAUX = ("DEBUG ", "INFO ", "WARNING ", "ERROR ", "CRITICAL ")
RESUME = """9 serveurs valides, 0 invalides
par env : prod 7, staging 2
par rôle : web 3, api 3, worker 1, cache 1, db 1
prod/web : web-1, web-2
"""

r = run("-v")
journal = r.stderr.splitlines()
check(all(ligne.startswith(NIVEAUX) for ligne in journal), True)
check(any(ligne.startswith("INFO ") for ligne in journal), True)
check(any(ligne.startswith("DEBUG ") for ligne in journal), False)
check(r.stdout, RESUME)

Avec `-vv`, au moins une ligne `DEBUG` s'ajoute. Si le point de contrôle de 12:15 a retiré `-vv` du socle, cette vérification passe en « pour aller plus loin ».

In [ ]:
r = run("-vv")
journal = r.stderr.splitlines()
check(all(ligne.startswith(NIVEAUX) for ligne in journal), True)
check(any(ligne.startswith("DEBUG ") for ligne in journal), True)
check(r.stdout, RESUME)

### 2. `exit_code` — fin 12:15

`exit_code(statuses, invalid_count)` reçoit une liste de statuts de services (`"ok"`, `"degraded"`, `"down"`) et le nombre de lignes invalides. Elle renvoie :

| Condition, évaluée dans l'ordre | Code |
|---|---|
| `invalid_count` non nul | `2` |
| au moins un statut `down` | `1` |
| sinon | `0` |

La fonction est pure : ni `print`, ni `sys.exit`, ni lecture de fichier. Aucun service n'est encore contrôlé (le contrôle réseau arrive en v0.8) : `main()` l'appelle avec une liste vide.

In [ ]:
%run -n fleetcheck.py


**Justifier** — Une ligne d'inventaire invalide l'emporte sur un service `down` : `exit_code(["down"], 1)` vaut `2`. Pourquoi cet ordre ?

In [ ]:
# Ma réponse :

Vérification : `1 2 0`, puis `2` quand une ligne invalide et un `down` coexistent.

In [ ]:
%run -n fleetcheck.py
check((exit_code(["ok", "down"], 0), exit_code(["ok"], 1), exit_code([], 0)), (1, 2, 0))
check(exit_code(["down"], 1), 2)

### 3. Codes de sortie — fin 12:25

Erreurs fatales : log `ERROR`, puis `sys.exit(2)`, sans écrire `report.json`.

| Cas | Message de la référence |
|---|---|
| Inventaire absent | `ERROR inventaire illisible : [Errno 2] No such file or directory: 'nope.csv'` |
| Config absente ou JSON invalide | `ERROR config illisible (config.json) : ...` |
| Aucune ligne valide dans l'inventaire | `ERROR aucune ligne valide dans <fichier>` |
| `--env` ne garde aucun serveur | `ERROR aucun serveur pour --env <valeur>` |

Fin de `main()` : `sys.exit(exit_code([], len(invalid)))`. Une ligne invalide donne donc `2`, mais le rapport est écrit et le résumé affiché : les serveurs valides restent traités.

Vérification des erreurs fatales : stdout vide, une ligne `ERROR` en dernier sur stderr, code `2`, pas de `report.json`. La cellule supprime `report.json` avant chaque cas, retire `config.json` le temps d'un run et crée un inventaire sans ligne valide ; elle remet tout en place.

In [ ]:
import shutil
from pathlib import Path

def fatal(*args):
    Path("report.json").unlink(missing_ok=True)
    r = run(*args)
    return r.stdout, r.stderr.splitlines()[-1:], r.returncode, Path("report.json").exists()

check(fatal("--inventory", "nope.csv"),
      ("", ["ERROR inventaire illisible : [Errno 2] No such file or directory: 'nope.csv'"], 2, False))

shutil.move("config.json", "config.bak")
try:
    resultat = fatal()
finally:
    shutil.move("config.bak", "config.json")
check(resultat, ("", ["ERROR config illisible (config.json) : [Errno 2] No such file or directory: 'config.json'"], 2, False))

Path("aucune-valide.csv").write_text("name,env,ip,port,role\napi-3,prod,127.0.0.1,abc,api\n", encoding="utf-8")
try:
    resultat = fatal("--inventory", "aucune-valide.csv")
finally:
    Path("aucune-valide.csv").unlink()
check(resultat, ("", ["ERROR aucune ligne valide dans aucune-valide.csv"], 2, False))

## Critère final

Quatre lancements, l'équivalent de ce qu'une CI exécuterait :

| Lancement | stdout | stderr | Code |
|---|---|---|---|
| `run()` | le résumé | rien | `0` |
| `run("--inventory", "servers-broken.csv")` | le résumé | les 3 `WARNING` | `2` |
| `run("--env", "prd")` | rien | `ERROR aucun serveur pour --env prd` | `2` |
| `run("--format", "json")` | un JSON valide | rien | `0` |

**Prédire** — Dans un terminal bash : `python3 fleetcheck.py --inventory servers-broken.csv --format json | jq . && echo OK`. `fleetcheck` sort en `2`. `OK` s'affiche-t-il ?

In [ ]:
# Ma réponse :

In [ ]:
r = run()
check((r.stdout, r.stderr, r.returncode), (RESUME, "", 0))

In [ ]:
r = run("--inventory", "servers-broken.csv")
check((r.stderr.count("WARNING ligne"), r.returncode), (3, 2))

In [ ]:
r = run("--env", "prd")
check((r.stdout, r.stderr, r.returncode), ("", "ERROR aucun serveur pour --env prd\n", 2))

In [ ]:
import json

r = run("--format", "json")
check((json.loads(r.stdout)["summary"], r.stderr, r.returncode), ({"total": 9, "invalid": 0}, "", 0))

En v0.4, `--env prd` affichait `0 serveurs valides, 0 invalides` et sortait en `0` : une faute de frappe dans un job de CI laissait le pipeline vert sans rien contrôler.

## Pour aller plus loin

- `-vv` (niveau `DEBUG`), si le point de contrôle de 12:15 l'a retiré du socle : sa vérification est à l'étape 1. Exemple de la référence, une ligne par ligne d'inventaire lue :

```text
DEBUG ligne 2 valide : web-1
DEBUG ligne 3 valide : web-2
...
DEBUG ligne 10 valide : db-1
INFO 9 serveur(s) valide(s), 0 ligne(s) invalide(s) dans servers.csv
INFO rapport écrit : report.json
```

- Étape 3 non finie à 12:25 : terminez-la chez vous à partir de la référence v0.5, publiée à 13:00.